# 03 — Webapp: Plot Boundary Detector + Heuristic Land-Use (Colab, Gradio)

Pick a tile → see detected plots traced on the aerial photo, **colour-coded by guessed land-use** (field / pond / built-up / bare). The classifier is a colour-and-texture *heuristic*, not a trained model — class labels don't exist in the GT shapefiles yet. Once a v3 labelled classifier exists, this cell swaps it in.

Prereqs:
- `02_train_colab.ipynb` has produced `MyDrive/aigeolab_train/checkpoints/best.pt`.
- `01_prep_colab.ipynb` has produced `MyDrive/aigeolab_train/{tiles, manifest.csv}`.

**Runtime: GPU.** Run All. The last cell prints a `https://*.gradio.live` URL.

In [ ]:
# --- Cell 1: install deps + mount Drive + clone repo ---
!pip install -q gradio rasterio segmentation-models-pytorch albumentations opencv-python-headless pyyaml pyshp

from google.colab import drive
drive.mount('/content/drive')

import os, subprocess, sys
REPO_URL = 'https://github.com/tahmid013/AIGEOLAB_OFFICE.git'
REPO_DIR = '/content/AIGEOLAB_OFFICE'
if os.path.isdir(REPO_DIR):
    print(subprocess.run(['git', '-C', REPO_DIR, 'pull'], capture_output=True, text=True).stdout)
else:
    print(subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], capture_output=True, text=True).stdout)
os.chdir(REPO_DIR)
sys.path.insert(0, os.path.join(REPO_DIR, 'src'))


In [ ]:
# --- Cell 2: load model + manifest ---
import yaml, csv
import torch, numpy as np, rasterio, cv2
from pathlib import Path
from inference import (load_model, predict_heatmap, extract_polygons,
                       classify_polygons, draw_polygons_classified, polygon_stats,
                       CLASS_COLOURS)

with open('config.yaml') as f: CFG = yaml.safe_load(f)
STAGE = Path(CFG['paths']['colab']['staging_root'])
CKPT  = STAGE / 'checkpoints' / 'best.pt'
assert CKPT.exists(), f'Checkpoint not found: {CKPT}. Run 02_train_colab.ipynb first.'

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
model, model_cfg, meta = load_model(CKPT, device=DEVICE)
PATCH = model_cfg['dataset']['patch_size']
STRIDE = PATCH // 2
print(f'Loaded model from {CKPT}')
print(f'  trained {meta["epoch"]} epochs, val_iou={meta["val_iou"]:.4f}')

manifest = []
with open(STAGE / 'manifest.csv') as fh:
    for r in csv.DictReader(fh):
        manifest.append(r)

tile_options = []
for r in manifest:
    mouzas = r['mouza_label_stems'].replace('|', ', ').replace('03.251222__', '').replace('04.251226__', '')
    tile_options.append((f"Tile {r['tile_x_km']}-{r['tile_y_km']}  ·  covers: {mouzas}", r['tile_tif_relpath']))
print(f'{len(tile_options)} tiles in dropdown')


In [ ]:
# --- Cell 3: inference pipeline wrapped for the app ---
PIXEL_SIZE_M = 0.1
CROP_SIZE    = 2048
DEFAULT_THRESHOLD = 0.4
DEFAULT_MIN_AREA  = 400
DEFAULT_RECT_IOU  = 0.86

def load_tile_crop(tif_relpath, top_frac=0.5, left_frac=0.5, size=CROP_SIZE):
    tif_path = STAGE / tif_relpath
    with rasterio.open(tif_path) as ds:
        H, W = ds.height, ds.width
        max_top = max(0, H - size); max_left = max(0, W - size)
        top  = int(max_top  * top_frac)
        left = int(max_left * left_frac)
        win = rasterio.windows.Window(left, top, min(size, W), min(size, H))
        arr = ds.read([1, 2, 3], window=win)
    return np.transpose(arr, (1, 2, 0)).astype(np.uint8)

def analyse(tile_relpath, top_frac, left_frac, threshold, min_area, snap_to_rect):
    rgb  = load_tile_crop(tile_relpath, top_frac=top_frac, left_frac=left_frac)
    prob = predict_heatmap(model, rgb, patch=PATCH, stride=STRIDE, device=DEVICE)
    polys = extract_polygons(prob,
                              threshold=threshold,
                              min_area_px=int(min_area),
                              approx_eps_frac=0.015,
                              regularise_to_rect=bool(snap_to_rect),
                              rect_iou_threshold=DEFAULT_RECT_IOU)
    classes = classify_polygons(rgb, polys)
    overlay = draw_polygons_classified(rgb, polys, classes, thickness=3, fill_alpha=0.20, label_classes=True)
    stats   = polygon_stats(polys, pixel_size_m=PIXEL_SIZE_M, classes=classes)

    # Raw probability heatmap as a 3rd panel
    heat = (prob * 255).astype(np.uint8)
    heat_rgb = cv2.applyColorMap(heat, cv2.COLORMAP_VIRIDIS)[:, :, ::-1]
    heat_blend = (rgb.astype(np.float32) * 0.45 + heat_rgb.astype(np.float32) * 0.55).clip(0, 255).astype(np.uint8)

    # Class breakdown lines
    by_class_lines = []
    for cls in ['cultivated field', 'vegetation/fallow', 'pond / water', 'built-up / house', 'bare ground/path', 'uncategorised']:
        d = stats['by_class'].get(cls)
        if not d: continue
        col = CLASS_COLOURS[cls]
        swatch = f'<span style="display:inline-block;width:14px;height:14px;background:rgb({col[0]},{col[1]},{col[2]});border-radius:3px;vertical-align:middle;margin-right:6px"></span>'
        by_class_lines.append(f'- {swatch}**{cls}** — {d["count"]} plots, {d["total_m2"]:,.0f} m² ({d["total_m2"]/10000:.2f} ha)')
    by_class_md = '\n'.join(by_class_lines) if by_class_lines else '_No plots above the size threshold yet._'

    md = f'''### Detection summary

- **Plots detected:** {stats["n_plots"]}
- **Total area:** {stats["total_area_m2"]:,.0f} m² ({stats["total_area_m2"]/10000:.2f} hectares)
- **Average plot:** {stats["mean_area_m2"]:,.0f} m²  ·  **Median plot:** {stats["median_area_m2"]:,.0f} m²
- **Photo scale:** 10 cm per pixel  ·  {CROP_SIZE//10} m × {CROP_SIZE//10} m visible area

**Breakdown by guessed land-use:**

{by_class_md}

_Class labels here are a colour/texture heuristic, not from training data. They give a rough sense of what each plot looks like._
_Boundary detection is confident above **{int(threshold*100)}%**; plots smaller than **{int(min_area * PIXEL_SIZE_M ** 2):,} m²** are filtered as noise._
'''
    return rgb, overlay, heat_blend, md

print('Pipeline ready.')


In [ ]:
# --- Cell 4: Gradio UI ---
import gradio as gr

CSS = '''
.gradio-container {max-width: 1400px !important}
h1 {font-size: 1.7rem !important}
h3 {margin-top: 0.5em !important}
'''

LEGEND_HTML = '<div style="display:flex;gap:18px;flex-wrap:wrap;margin:8px 0 0">' + ''.join(
    f'<span style="display:inline-flex;align-items:center;gap:6px"><span style="display:inline-block;width:14px;height:14px;background:rgb({c[0]},{c[1]},{c[2]});border-radius:3px"></span>{k}</span>'
    for k, c in CLASS_COLOURS.items()
) + '</div>'

with gr.Blocks(title='AIGEOLAB · Plot Boundary Detector', css=CSS, theme=gr.themes.Soft()) as app:
    gr.Markdown('# Plot Boundary Detector — Bangladesh')
    gr.Markdown(
        'Pick an aerial photo. The model traces every plot it can see and tags each one '
        'with a likely land-use type (field / pond / building / bare). Use the sliders if you '
        'want more or fewer detections.'
    )
    gr.HTML(LEGEND_HTML)

    with gr.Row():
        with gr.Column(scale=2):
            tile_dd = gr.Dropdown(choices=tile_options, label='1.  Select an aerial photo',
                                   value=tile_options[0][1] if tile_options else None, interactive=True)
        with gr.Column(scale=1):
            top_slider  = gr.Slider(0, 1, value=0.5, step=0.1, label='Vertical crop  (top 0 — bottom 1)')
            left_slider = gr.Slider(0, 1, value=0.5, step=0.1, label='Horizontal crop (left 0 — right 1)')

    with gr.Accordion('Advanced — detection sensitivity', open=False):
        thr_slider  = gr.Slider(0.1, 0.8, value=DEFAULT_THRESHOLD, step=0.05, label='Boundary confidence (lower = more detections)')
        area_slider = gr.Slider(50, 5000, value=DEFAULT_MIN_AREA, step=50, label='Minimum plot area (pixels)')
        snap_chk    = gr.Checkbox(value=True, label='Snap near-rectangular plots to clean rectangles')

    btn = gr.Button('2.  Analyze this photo', variant='primary', size='lg')

    summary = gr.Markdown()

    with gr.Row():
        img_orig = gr.Image(label='Aerial photo',                         show_label=True, height=520)
        img_out  = gr.Image(label='Detected plots (coloured by land-use)', show_label=True, height=520)
    with gr.Accordion('Show raw model heatmap (where the model thinks edges are)', open=False):
        img_heat = gr.Image(label='Boundary probability heatmap',           show_label=True, height=520)

    btn.click(
        analyse,
        inputs=[tile_dd, top_slider, left_slider, thr_slider, area_slider, snap_chk],
        outputs=[img_orig, img_out, img_heat, summary],
    )

    gr.Markdown(
        '---\n'
        f'<small>Model: U-Net (ResNet-34) · trained {meta["epoch"]} epochs · val IoU (masked) {meta["val_iou"]:.3f}'
        f' · {len(manifest)} tiles staged · 10 cm/px imagery · land-use classifier is a colour/texture heuristic (v3 will replace it once class labels exist)</small>'
    )


In [ ]:
# --- Cell 5: launch the app (prints a public gradio.live URL) ---
app.queue(default_concurrency_limit=1).launch(share=True, inline=False, debug=False)
